In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_a = pd.read_csv('/content/drive/MyDrive/arkansas_partie2.csv')
df_c = pd.read_csv('/content/drive/MyDrive/california_partie2.csv')

print('Arkansas shape:', df_a.shape)
print('California shape:', df_c.shape)

In [ ]:
# Vérifier colonnes climat
climat_cols = [c for c in df_a.columns if 'precip' in c or 'tmax' in c or 'tmin' in c]
print('Nombre colonnes climat:', len(climat_cols))
print(climat_cols[:6])

In [ ]:
print('Arkansas classes:', df_a['crop_name'].value_counts())
print('\nCalifornia classes:', df_c['crop_name'].value_counts())

## Arkansas — S2 + Climat

In [ ]:
# X et y Arkansas
df_a_x = df_a.drop(columns=['crop_label', 'crop_name'])
df_a_y = df_a[['crop_name', 'crop_label']].copy()
df_a_y.loc[df_a['crop_name'] == 'Others', 'crop_label'] = 0
df_a_y.loc[df_a['crop_label'] == 5,       'crop_label'] = 4
y_a = df_a_y['crop_label'].values

In [ ]:
# X_time : bandes S2 (36, 10)
X_time_a = df_a_x[[col for col in df_a_x.columns if '_T' in col and col.startswith('B')]].values
X_time_a = X_time_a.reshape(-1, 36, 10)

# X_static : 108 colonnes climat
climat_cols = [c for c in df_a_x.columns if 'precip' in c or 'tmax' in c or 'tmin' in c]
X_static_a = df_a_x[climat_cols].values

print('X_time shape :', X_time_a.shape)    # (N, 36, 10)
print('X_static shape:', X_static_a.shape) # (N, 108)

# Normalisation
X_max = X_time_a.max(axis=(0, 1), keepdims=True)
with np.errstate(invalid='ignore', divide='ignore'):
    X_time_a = np.where(X_max == 0, 0, X_time_a / X_max)

X_static_max = np.abs(X_static_a).max(axis=0)
X_static_max[X_static_max == 0] = 1
X_static_a = X_static_a / X_static_max

In [ ]:
mask_time_a = (X_time_a != 0).astype(float)

In [ ]:
# Split Arkansas
X_time_train_a, X_time_test_a, X_static_train_a, X_static_test_a, \
y_train_a, y_test_a, mask_train_a, mask_test_a = train_test_split(
    X_time_a, X_static_a, y_a, mask_time_a,
    test_size=0.85, stratify=y_a, random_state=42
)

X_time_train_a, X_time_val_a, X_static_train_a, X_static_val_a, \
y_train_a, y_val_a, mask_train_a, mask_val_a = train_test_split(
    X_time_train_a, X_static_train_a, y_train_a, mask_train_a,
    test_size=0.2, stratify=y_train_a, random_state=42
)

print('Train:', X_time_train_a.shape, '| Val:', X_time_val_a.shape, '| Test:', X_time_test_a.shape)

In [ ]:
# Class weights Arkansas
classes_a = np.unique(y_train_a)
weights_a = compute_class_weight('balanced', classes=classes_a, y=y_train_a)
class_weight_dict_a = dict(zip(classes_a, weights_a))
print('Class weights:', class_weight_dict_a)

In [ ]:
def CNNBlock(x):
    shortcut = x
    dim = x.shape[-1]
    x = layers.Conv1D(filters=dim, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv1D(filters=dim, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

In [ ]:
def TransformerBlock(x, num_heads=5):
    dim = x.shape[-1]
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)
    ff = layers.Dense(dim, activation='relu')(x)
    ff = layers.Dense(dim)(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)
    return x

In [ ]:
def ALPE(x, mask):
    x = layers.Multiply()([x, mask])
    x = layers.Conv1D(filters=x.shape[-1], kernel_size=3, padding='same')(x)
    C = x.shape[-1]
    import math
    gamma, b = 2, 1
    t = int(math.log2(C) / gamma + b / gamma)
    k = t if t % 2 else t + 1
    weights = layers.GlobalAveragePooling1D()(x)
    weights = layers.Reshape((C, 1))(weights)
    weights = layers.Conv1D(filters=1, kernel_size=k, padding='same', use_bias=False)(weights)
    weights = layers.Activation('sigmoid')(weights)
    weights = layers.Reshape((1, C))(weights)
    x = layers.Multiply()([x, weights])
    return x

In [ ]:
def CTFusion(x):
    cnn_out   = CNNBlock(x)
    trans_out = TransformerBlock(x)
    fused = layers.Concatenate()([cnn_out, trans_out])
    fused = layers.Dense(x.shape[-1])(fused)
    return fused

In [ ]:
def build_MCTNet(input_shape=(36, 10), static_shape=(108,), num_classes=5):
    # Input temporel S2
    inputs = tf.keras.Input(shape=input_shape)
    mask   = tf.keras.Input(shape=input_shape)

    x = ALPE(inputs, mask)

    x = CTFusion(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = CTFusion(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = CTFusion(x)
    x = layers.GlobalMaxPooling1D()(x)

    # Input Climat (108 features)
    inputs_static = tf.keras.Input(shape=static_shape)
    x_static = layers.Dense(64, activation='relu')(inputs_static)
    x_static = layers.Dense(32, activation='relu')(x_static)

    # Fusion
    x = layers.Concatenate()([x, x_static])

    # Classifier
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(
        inputs=[inputs, mask, inputs_static],
        outputs=outputs
    )
    return model

In [ ]:
model_a = build_MCTNet(input_shape=(36, 10), static_shape=(108,), num_classes=5)
model_a.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_a.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=10, min_lr=1e-6, verbose=1
    )
]

history_a = model_a.fit(
    [X_time_train_a, mask_train_a, X_static_train_a], y_train_a,
    validation_data=([X_time_val_a, mask_val_a, X_static_val_a], y_val_a),
    epochs=200,
    batch_size=32,
    class_weight=class_weight_dict_a,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
y_pred_a = np.argmax(model_a.predict([X_time_test_a, mask_test_a, X_static_test_a]), axis=1)

oa_a  = accuracy_score(y_test_a, y_pred_a)
f1_a  = f1_score(y_test_a, y_pred_a, average='macro')
kap_a = cohen_kappa_score(y_test_a, y_pred_a)

print('=' * 40)
print('ARKANSAS — S2 + Climat')
print(f'OA    : {oa_a:.4f}')
print(f'F1    : {f1_a:.4f}')
print(f'Kappa : {kap_a:.4f}')
print('=' * 40)

labels_a = ['Others', 'Corn', 'Cotton', 'Rice', 'Soybeans']
print(classification_report(y_test_a, y_pred_a, target_names=labels_a))

In [ ]:
cm_a = confusion_matrix(y_test_a, y_pred_a)
cm_a_norm = cm_a.astype(float) / cm_a.sum(axis=1, keepdims=True)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_a_norm, display_labels=labels_a)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Arkansas (S2 + Climat)')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_a.history['loss'],     label='Train')
ax1.plot(history_a.history['val_loss'], label='Val')
ax1.set_title('Loss — Arkansas')
ax1.legend()
ax1.grid(True)
ax2.plot(history_a.history['accuracy'],     label='Train')
ax2.plot(history_a.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy — Arkansas')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

## California — S2 + Climat

In [ ]:
df_c_x = df_c.drop(columns=['crop_label', 'crop_name'])
label_map_c = {'Grapes': 0, 'Rice': 1, 'Alfalfa': 2, 'Almonds': 3, 'Pistachios': 4, 'Others': 5}
y_c = df_c['crop_name'].map(label_map_c).values

In [ ]:
X_time_c = df_c_x[[col for col in df_c_x.columns if '_T' in col and col.startswith('B')]].values
X_time_c = X_time_c.reshape(-1, 36, 10)

climat_cols_c = [c for c in df_c_x.columns if 'precip' in c or 'tmax' in c or 'tmin' in c]
X_static_c = df_c_x[climat_cols_c].values

print('X_time shape :', X_time_c.shape)
print('X_static shape:', X_static_c.shape)

X_max_c = X_time_c.max(axis=(0, 1), keepdims=True)
with np.errstate(invalid='ignore', divide='ignore'):
    X_time_c = np.where(X_max_c == 0, 0, X_time_c / X_max_c)

X_static_max_c = np.abs(X_static_c).max(axis=0)
X_static_max_c[X_static_max_c == 0] = 1
X_static_c = X_static_c / X_static_max_c

In [ ]:
mask_time_c = (X_time_c != 0).astype(float)

In [ ]:
X_time_train_c, X_time_test_c, X_static_train_c, X_static_test_c, \
y_train_c, y_test_c, mask_train_c, mask_test_c = train_test_split(
    X_time_c, X_static_c, y_c, mask_time_c,
    test_size=0.85, stratify=y_c, random_state=42
)

X_time_train_c, X_time_val_c, X_static_train_c, X_static_val_c, \
y_train_c, y_val_c, mask_train_c, mask_val_c = train_test_split(
    X_time_train_c, X_static_train_c, y_train_c, mask_train_c,
    test_size=0.2, stratify=y_train_c, random_state=42
)

print('Train:', X_time_train_c.shape, '| Val:', X_time_val_c.shape, '| Test:', X_time_test_c.shape)

In [ ]:
classes_c = np.unique(y_train_c)
weights_c = compute_class_weight('balanced', classes=classes_c, y=y_train_c)
class_weight_dict_c = dict(zip(classes_c, weights_c))
print('Class weights California:', class_weight_dict_c)

In [ ]:
model_c = build_MCTNet(input_shape=(36, 10), static_shape=(108,), num_classes=6)
model_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_c.summary()

In [ ]:
history_c = model_c.fit(
    [X_time_train_c, mask_train_c, X_static_train_c], y_train_c,
    validation_data=([X_time_val_c, mask_val_c, X_static_val_c], y_val_c),
    epochs=200,
    batch_size=32,
    class_weight=class_weight_dict_c,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
y_pred_c = np.argmax(model_c.predict([X_time_test_c, mask_test_c, X_static_test_c]), axis=1)

oa_c  = accuracy_score(y_test_c, y_pred_c)
f1_c  = f1_score(y_test_c, y_pred_c, average='macro')
kap_c = cohen_kappa_score(y_test_c, y_pred_c)

print('=' * 40)
print('CALIFORNIA — S2 + Climat')
print(f'OA    : {oa_c:.4f}')
print(f'F1    : {f1_c:.4f}')
print(f'Kappa : {kap_c:.4f}')
print('=' * 40)

labels_c = ['Grapes', 'Rice', 'Alfalfa', 'Almonds', 'Pistachios', 'Others']
print(classification_report(y_test_c, y_pred_c, target_names=labels_c))

In [ ]:
cm_c = confusion_matrix(y_test_c, y_pred_c)
cm_c_norm = cm_c.astype(float) / cm_c.sum(axis=1, keepdims=True)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_c_norm, display_labels=labels_c)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — California (S2 + Climat)')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_c.history['loss'],     label='Train')
ax1.plot(history_c.history['val_loss'], label='Val')
ax1.set_title('Loss — California')
ax1.legend()
ax1.grid(True)
ax2.plot(history_c.history['accuracy'],     label='Train')
ax2.plot(history_c.history['val_accuracy'], label='Val')
ax2.set_title('Accuracy — California')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print('=' * 50)
print('RÉSUMÉ — Configuration S2 + Climat')
print('=' * 50)
print(f'Arkansas   → OA: {oa_a:.4f} | F1: {f1_a:.4f} | Kappa: {kap_a:.4f}')
print(f'California → OA: {oa_c:.4f} | F1: {f1_c:.4f} | Kappa: {kap_c:.4f}')
print('=' * 50)